In [1]:
import torch
import torch.nn as nn
import pandas as pd
import re
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np

class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def simple_tokenize(text):
        """
        Standard Python regex tokenizer.
        Adds spaces around punctuation so they become separate tokens.
        """
        text = str(text).lower().strip()
        # Create space around punctuation: "hello!" -> "hello !"
        text = re.sub(r"([?.!,¿¡])", r" \1 ", text)
        # Remove extra spaces
        text = re.sub(r'[" "]+', " ", text)
        return text.split()

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4
        for sentence in sentence_list:
            for word in self.simple_tokenize(sentence):
                frequencies[word] = frequencies.get(word, 0) + 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.simple_tokenize(text)
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokenized_text]

class EngSpaDataset(Dataset):
    def __init__(self, df, freq_threshold=2):
        # Your dataframe info showed 118,964 entries
        self.df = df.dropna().reset_index(drop=True)
        
        self.eng_vocab = Vocabulary(freq_threshold)
        self.spa_vocab = Vocabulary(freq_threshold)
        
        # Build using standard Python lists for speed
        self.eng_vocab.build_vocabulary(self.df["english"].tolist())
        self.spa_vocab.build_vocabulary(self.df["spanish"].tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        eng_sent = self.df.iloc[index]["english"]
        spa_sent = self.df.iloc[index]["spanish"]
        
        num_eng = [1] + self.eng_vocab.numericalize(eng_sent) + [2] # 1=<SOS>, 2=<EOS>
        num_spa = [1] + self.spa_vocab.numericalize(spa_sent) + [2]
        
        return torch.tensor(num_eng), torch.tensor(num_spa)

In [2]:
df = pd.read_csv("./data/data.csv")
df['english'] = df['english'].astype(str)
df['spanish'] = df['spanish'].astype(str)

In [3]:
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embed_size, nhead, num_layers, max_len):
        super().__init__()
        self.embed_size = embed_size
        self.src_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, embed_size)
        
        # Simple learnable positional embedding for offline simplicity
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len, embed_size))
        
        self.transformer = nn.Transformer(
            d_model=embed_size, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            batch_first=True
        )
        self.fc_out = nn.Linear(embed_size, tgt_vocab_size)

    def forward(self, src, tgt):
        src_seq_len = src.shape[1]
        tgt_seq_len = tgt.shape[1]
        
        # 1. CREATE MASK FIRST (Using the raw integer src)
        # This creates a (Batch, Seq_Len) 2D boolean tensor
        src_padding_mask = (src == 0) 
        
        # 2. NOW transform src and tgt into embeddings
        src_emb = self.src_embedding(src) + self.pos_embedding[:, :src_seq_len, :]
        tgt_emb = self.tgt_embedding(tgt) + self.pos_embedding[:, :tgt_seq_len, :]
        
        # 3. Generate the causal mask for the decoder
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt_seq_len).to(src.device)
        
        # 4. Pass them to the transformer
        # Note: src_key_padding_mask MUST be 2D (Batch, Seq_Len)
        out = self.transformer(
            src_emb, 
            tgt_emb, 
            tgt_mask=tgt_mask, 
            src_key_padding_mask=src_padding_mask
        )
        
        return self.fc_out(out)

In [4]:
# Assuming 'df' is your pandas DataFrame
dataset = EngSpaDataset(df, freq_threshold=2)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=0, batch_first=True)
    return src_batch, tgt_batch

loader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerModel(
    src_vocab_size=len(dataset.eng_vocab),
    tgt_vocab_size=len(dataset.spa_vocab),
    embed_size=256, nhead=8, num_layers=3, max_len=100
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Single Epoch Training Example
model.train()
for batch_idx, (src, tgt) in enumerate(loader):
    src, tgt = src.to(device), tgt.to(device)
    
    # Target input (no last token), Target output (no first token)
    output = model(src, tgt[:, :-1])
    
    output = output.reshape(-1, output.shape[2])
    tgt_loss = tgt[:, 1:].reshape(-1)
    
    loss = criterion(output, tgt_loss)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if batch_idx % 100 == 0:
        print(f"Batch {batch_idx} Loss: {loss.item():.4f}")

Batch 0 Loss: 9.9187
Batch 100 Loss: 5.7434
Batch 200 Loss: 5.0456
Batch 300 Loss: 4.7970
Batch 400 Loss: 4.5836
Batch 500 Loss: 4.3883
Batch 600 Loss: 4.2792
Batch 700 Loss: 4.0517
Batch 800 Loss: 3.9618
Batch 900 Loss: 4.1123
Batch 1000 Loss: 3.6388
Batch 1100 Loss: 3.6334
Batch 1200 Loss: 3.7063
Batch 1300 Loss: 3.3850
Batch 1400 Loss: 3.5290
Batch 1500 Loss: 3.2121
Batch 1600 Loss: 3.1779
Batch 1700 Loss: 3.1139
Batch 1800 Loss: 3.3039


In [5]:
def translate_sentence(model, sentence, eng_vocab, spa_vocab, device, max_length=50):
    model.eval() # Set to evaluation mode
    
    # 1. Tokenize and Numericalize the English sentence
    # Use the same regex logic used in training
    import re
    tokens = str(sentence).lower().strip()
    tokens = re.sub(r"([?.!,¿¡])", r" \1 ", tokens)
    tokens = tokens.split()
    
    # Convert to IDs and add SOS/EOS
    src_indices = [1] + [eng_vocab.stoi.get(token, 3) for token in tokens] + [2]
    src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device) # Shape: (1, seq_len)
    
    # 2. Start the translation with the <SOS> token (index 1)
    outputs = [1] 
    
    for i in range(max_length):
        tgt_tensor = torch.LongTensor(outputs).unsqueeze(0).to(device)
        
        with torch.no_grad():
            # The model predicts the next word based on English and what it said so far
            output = model(src_tensor, tgt_tensor)
            
        # Take the word with the highest probability from the last position
        best_guess = output.argmax(2)[:, -1].item()
        outputs.append(best_guess)
        
        # If the model predicts <EOS> (index 2), stop translating
        if best_guess == 2:
            break
            
    # 3. Convert indices back to Spanish words
    translated_words = [spa_vocab.itos.get(idx, "<UNK>") for idx in outputs]
    
    # Remove <SOS> and <EOS> for the final display
    return " ".join(translated_words[1:-1])

In [7]:
# Select a random sentence or type your own
test_sentence = "i am very happy"

# Use the vocabs from your dataset object
prediction = translate_sentence(
    model, 
    test_sentence, 
    dataset.eng_vocab, 
    dataset.spa_vocab, 
    device
)

print(f"English: {test_sentence}")
print(f"Prediction: {prediction}")

English: i am very happy
Prediction: estoy muy feliz .
